**MediaEval 2025 NewsImage Generation Large Task**

Group: CVG-IBA

Reference file titled *"newsarticles_with_prompts.xlsx"* is required that was generated using the notebook titled *"MediaEval2025_prompt_generation_PromptForge_LARGE.ipynb"*

In [12]:
# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# --- Install dependencies ---
!pip install -U xformers --index-url https://download.pytorch.org/whl/cu121
!pip install -q openpyxl #pandas pyngrok

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 95.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 51.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 117.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1

In [4]:
# --- Clone and setup ComfyUI ---
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt

/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 24051, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 24051 (delta 43), reused 17 (delta 16), pack-reused 23981 (from 6)
Receiving objects: 100% (24051/24051), 72.68 MiB | 16.16 MiB/s, done.
Resolving deltas: 100% (16148/16148), done.
/content/ComfyUI
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 94.5 MB/

In [5]:
# --- Import dependencies ---

import os, shutil, subprocess, time, json, random, requests
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import hf_hub_download
from PIL import Image

In [6]:
# --- HuggingFace token ---
# Replace this with your own token
from huggingface_hub import login

# Current token which will expire in a month, for reference run only
HF_TOKEN = "[ENTER_YOUR_TOKEN_HERE]"

# Login
login(token=HF_TOKEN)

In [7]:
# --- Download from Hugging Face & move to ComfyUI ---
model_path = "/content/ComfyUI/models/checkpoints"
os.makedirs(model_path, exist_ok=True)

# Juggernaut XI
repo_id = "RunDiffusion/Juggernaut-XI-v11"
filename = "Juggernaut-XI-byRunDiffusion.safetensors"

print(f"Downloading {filename} from {repo_id}...")
downloaded_model = hf_hub_download(repo_id=repo_id, filename=filename)
shutil.copy(downloaded_model, os.path.join(model_path, filename))

print(f"✅ Model ready at {model_path}/{filename}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Juggernaut-XI-byRunDiffusion.safetensors:   0%|          | 0.00/7.11G [00:00<?, ?B/s]

✅ Model ready at /content/ComfyUI/models/checkpoints/Juggernaut-XI-byRunDiffusion.safetensors


In [16]:
# --- Launch ComfyUI server ---
log_file = "/content/ComfyUI/launch_log.txt"
with open(log_file, "w") as f:
    process = subprocess.Popen(
        ["python3", "main.py", "--dont-print-server"],
        cwd="/content/ComfyUI",
        stdout=f, stderr=f
    )

def is_comfy_ready():
    try:
        return requests.get("http://localhost:8188").status_code == 200
    except:
        return False

for i in range(20):
    if is_comfy_ready():
        print("✅ ComfyUI is ready")
        break
    time.sleep(2)
else:
    with open(log_file) as f:
        print(f.read())
    raise RuntimeError("❌ ComfyUI did not start correctly.")

✅ ComfyUI is ready


In [20]:
# --- Loading data from Excel (full data) ---

full_path = "/content/drive/MyDrive/mediaeval2025/newsarticles_with_prompts.xlsx" #modify as per the location of the file
df_full = pd.read_excel(full_path)

# Ensure article_id is numeric
df_full["article_id"] = pd.to_numeric(df_full["article_id"], errors="coerce")

# --- Define ranges manually (for small range and repeat this step to generate batches) ---
ranges = [(1, 8500)] #This is set to generate images for all 8500 articles, may be modified to test on a small subset
#ranges = [(1, 20)] #This will generate images for the first 20 articles

# Filter rows within these ranges
df_subset = pd.concat([
    df_full[(df_full["article_id"] >= start) & (df_full["article_id"] <= end)]
    for start, end in ranges
])
df_subset = df_subset.reset_index(drop=True)

print("✅ Loaded", len(df_subset), "articles from specified ranges")

# Prompt lookup with **integer keys** so generate_image works unchanged
prompt_lookup = dict(zip(
    df_subset["article_id"].astype(int),
    df_subset["positive_prompts_concatinated"].astype(str)
))

# Negative prompts (constant string)
negative_prompt_text = "blurry, low quality, deformed, bad anatomy, distorted, watermark"

# Article IDs list for sequential loop
article_ids = df_subset["article_id"].astype(int).tolist()


✅ Loaded 8500 articles from specified ranges


In [17]:
# --- Main generation loop ---
# Helper function - truncate_prompt(): limit the length to 77 tokens + inject style keywords ---
# Helper function - build_prompt(): to build prompt from positive and negative prompts, seed and filename---
# Main task - generate_image(): Generate image in an iterative way, multiple images could be set for a single prompt
# Output resolution is set to 1384x784 to utilize native resolution advantage of the model, this will be reduced to the desired resolution in post processing later

MUST_HAVE_KEYWORDS = "digital art, illustration, painting, cartoon, sketch, non-photorealistic style"
MUST_HAVE_TOKENS = len(MUST_HAVE_KEYWORDS.split())  # ~8 tokens
MAX_TOKENS = 77 - MUST_HAVE_TOKENS
COMFYUI_SERVER = "http://127.0.0.1:8188"  # ComfyUI server URL

def truncate_prompt(text, max_tokens=MAX_TOKENS):
    words = text.split()
    return " ".join(words[:max_tokens])

def build_prompt(prompt_text, neg_text, seed, filename_prefix):
    return {
        "1": {"class_type": "CheckpointLoaderSimple",
              "inputs": {"ckpt_name": "Juggernaut-XI-byRunDiffusion.safetensors"}},
        "2": {"class_type": "CLIPTextEncode",
              "inputs": {"clip": ["1", 1], "text": prompt_text}},
        "3": {"class_type": "CLIPTextEncode",
              "inputs": {"clip": ["1", 1], "text": neg_text}},
        "4": {"class_type": "EmptyLatentImage",
              "inputs": {"batch_size": 1, "height": 784, "width": 1384, "channels": 4}},
        "5": {"class_type": "KSampler",
              "inputs": {"model": ["1", 0], "positive": ["2", 0], "negative": ["3", 0],
                         "latent_image": ["4", 0], "sampler_name": "euler",
                         "scheduler": "normal", "steps": 20, "cfg": 7.5,
                         "seed": seed, "denoise": 1.0}},
        "6": {"class_type": "VAEDecode",
              "inputs": {"samples": ["5", 0], "vae": ["1", 2]}},
        "7": {"class_type": "SaveImage",
              "inputs": {"images": ["6", 0],
                         "filename_prefix": filename_prefix,
                         "output_path": "/content/drive/MyDrive/comfyui_outputs/"}}
    }

def generate_image(article_id, j):
    """Generate a single non-photorealistic image sequentially"""

    # --- Print progress before sending ---
    print(f"⏳ Starting generation: article {article_id}, image {j+1}")

    if article_id not in prompt_lookup:
        return f"⚠️ Skipping {article_id}: no prompt found in Excel"

    # --- Style injection + truncation ---
    base_prompt = prompt_lookup[article_id]
    truncated = truncate_prompt(base_prompt)
    styled_prompt = f"{truncated}, {MUST_HAVE_KEYWORDS}"

    seed = random.randint(0, 2**32 - 1)
    filename_prefix = f"{article_id}_{seed}_{j+1}"
    prompt = build_prompt(styled_prompt, negative_prompt_text, seed, filename_prefix)

    try:
        response = requests.post(f"{COMFYUI_SERVER}/prompt", json={"prompt": prompt})
        result = response.json()

        if "error" in result:
            return f"❌ Failed [{article_id}-{j+1}]: {json.dumps(result, indent=2)}"

        # --- Wait for the file to appear ---
        out_dir = "/content/ComfyUI/output"
        expected_file = None
        timeout = 180  # max seconds to wait

        start = time.time()
        while time.time() - start < timeout:
            f_name = filename_prefix + "_00001_"
            for f in os.listdir(out_dir):
                if f.startswith(f_name):
                    expected_file = f
                    break
            if expected_file:
                break
            time.sleep(2)

        if expected_file:
            return f"✅ Done: article {article_id}, image {j+1}/4 (seed {seed}) → {expected_file}"
        else:
            return f"⚠️ Timeout waiting for output of {article_id}-{j+1}"

    except Exception as e:
        return f"❌ Exception for {article_id} image {j+1}: {str(e)}"


# --- Sequential run --- Modify the range(1) in the loop to generate multiple images per article ID if required
for article_id in article_ids:
    for j in range(1):
        status = generate_image(article_id, j)
        print(status)

⏳ Starting generation: article 1, image 1
✅ Done: article 1, image 1/4 (seed 1549511274) → 1_1549511274_1_00001_.png
⏳ Starting generation: article 2, image 1
✅ Done: article 2, image 1/4 (seed 4276497066) → 2_4276497066_1_00001_.png
⏳ Starting generation: article 3, image 1
✅ Done: article 3, image 1/4 (seed 797591264) → 3_797591264_1_00001_.png
⏳ Starting generation: article 4, image 1
✅ Done: article 4, image 1/4 (seed 3815820591) → 4_3815820591_1_00001_.png
⏳ Starting generation: article 5, image 1
✅ Done: article 5, image 1/4 (seed 3626433870) → 5_3626433870_1_00001_.png
⏳ Starting generation: article 6, image 1
✅ Done: article 6, image 1/4 (seed 3902981109) → 6_3902981109_1_00001_.png
⏳ Starting generation: article 7, image 1
✅ Done: article 7, image 1/4 (seed 783409213) → 7_783409213_1_00001_.png
⏳ Starting generation: article 8, image 1
✅ Done: article 8, image 1/4 (seed 2049340352) → 8_2049340352_1_00001_.png
⏳ Starting generation: article 9, image 1
✅ Done: article 9, image 1

In [18]:
# --- Post processing image to desired resolution of 460x260 ---

# Input and output directories
input_dir = "/content/ComfyUI/output"
output_dir = "/content/ComfyUI/output_resized"
os.makedirs(output_dir, exist_ok=True)

TARGET_SIZE = (460, 260)  # width, height

def center_crop(img, target_size):
    """Crops the image to target_size (width, height) from center"""
    target_w, target_h = target_size
    w, h = img.size
    if w < target_w or h < target_h:
        # If smaller, just return resized again (no crop possible)
        return img.resize(target_size, Image.LANCZOS)

    left = (w - target_w) // 2
    top = (h - target_h) // 2
    right = left + target_w
    bottom = top + target_h
    return img.crop((left, top, right, bottom))


# --- Resize loop ---
for filename in os.listdir(input_dir):
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        try:
            img = Image.open(input_path)
            # Step 1: Resize by factor of 3
            new_size = (img.width // 3, img.height // 3)
            img_resized = img.resize(new_size, Image.LANCZOS)

            # Step 2: If larger than 460x260, crop
            if img_resized.width > TARGET_SIZE[0] or img_resized.height > TARGET_SIZE[1]:
                img_resized = center_crop(img_resized, TARGET_SIZE)

            # Step 3: Save
            img_resized.save(output_path)

        except Exception as e:
            print(f"⚠️ Could not process {filename}: {e}")

print(f"✅ All images resized (1/3 scale) and cropped to max {TARGET_SIZE} saved to: {output_dir}")


✅ All images resized (1/3 scale) and cropped to max (460, 260) saved to: /content/ComfyUI/output_resized


In [19]:
# --- Archiving images and uploading to drive ---

output_folder = "/content/ComfyUI/output_resized"  # folder to zip
zip_base = "/content/0001-8500"  # base name without .zip
zip_filename = f"{zip_base}.zip"
drive_dest = "/content/drive/MyDrive/mediaeval2025/0001-8500.zip"  # must include .zip

# Create zip archive
shutil.make_archive(zip_base, 'zip', output_folder)

# Copy zip to Google Drive
shutil.copy(zip_filename, drive_dest)

print(f"✅ Zipped and copied to: {drive_dest}")

✅ Zipped and copied to: /content/drive/MyDrive/mediaeval2025/0001-8500.zip


In [ ]:
# Flush Drive if already mounted ---
drive_path = "/content/drive"
if os.path.ismount(drive_path):
    drive.flush_and_unmount()
    print("🔄 Flushed Google Drive.")

🔄 Flushed Google Drive.
